# Final Preparation
**Project:** F1 Constructor Sponsorship & Investment ROI Analytics  
**Objective:** Compute all KPIs, create pre-aggregated tables, and export Tableau-ready CSV files that power the final dashboard.  
**Input:** `data/processed/f1_modern_clean.csv`, `data/processed/pit_stops_clean.csv`, `data/processed/constructor_standings_final.csv`  
**Output:** Multiple CSV files in `data/processed/tableau/` optimised for dashboard consumption.  


## 1. Setup


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
print('Final load prep environment ready.')


Final load prep environment ready.


## 2. Load Processed Data


In [2]:
PROC = '../data/processed/'
df = pd.read_csv(f'{PROC}f1_modern_clean.csv', parse_dates=['race_date'])
pit = pd.read_csv(f'{PROC}pit_stops_clean.csv')
standings = pd.read_csv(f'{PROC}constructor_standings_final.csv')

print(f'Main: {df.shape} | Pit: {pit.shape} | Standings: {standings.shape}')


Main: (4626, 28) | Pit: (8360, 9) | Standings: (112, 6)


## 3. KPI Table 1 — Constructor Season Summary
This is the **primary Tableau data source**. One row per constructor per season with all KPIs pre-computed.


In [3]:
# Build season-level aggregates
season_summary = df.groupby(['constructor_name', 'constructor_nationality', 'year']).agg(
    total_points=('points', 'sum'),
    avg_finish_position=('positionOrder', 'mean'),
    result_volatility=('positionOrder', 'std'),
    finish_rate=('finished', 'mean'),
    avg_grid_position=('grid', 'mean'),
    avg_position_delta=('position_delta', 'mean'),
    total_races=('resultId', 'count'),
    total_laps=('laps', 'sum'),
    dnf_count=('finished', lambda x: (x == 0).sum()),
    best_finish=('positionOrder', 'min'),
    worst_finish=('positionOrder', 'max'),
    podiums=('positionOrder', lambda x: (x <= 3).sum()),
    wins=('positionOrder', lambda x: (x == 1).sum()),
).reset_index()

# Add championship position from standings
season_summary = season_summary.merge(
    standings[['constructor_name', 'year', 'position']].rename(columns={'position': 'championship_position'}),
    on=['constructor_name', 'year'], how='left'
)

# Calculate YoY Growth %
season_summary = season_summary.sort_values(['constructor_name', 'year'])
season_summary['prev_year_points'] = season_summary.groupby('constructor_name')['total_points'].shift(1)
season_summary['yoy_growth_pct'] = ((season_summary['total_points'] - season_summary['prev_year_points']) 
                                     / season_summary['prev_year_points'] * 100)

# Points per race efficiency
season_summary['points_per_race'] = season_summary['total_points'] / season_summary['total_races']

# Calculate momentum slope (rolling 3-year trend)
def calc_momentum(group):
    group = group.sort_values('year')
    slopes = []
    for i in range(len(group)):
        window = group.iloc[max(0, i-2):i+1]
        if len(window) >= 2:
            slope, _, _, _, _ = stats.linregress(window['year'], window['total_points'])
            slopes.append(slope)
        else:
            slopes.append(np.nan)
    group['momentum_slope'] = slopes
    return group

season_summary = season_summary.groupby('constructor_name', group_keys=False).apply(calc_momentum)

print(f'Constructor Season Summary: {season_summary.shape}')
print(f'Columns: {list(season_summary.columns)}')
print()
# Show 2024 season as sample
print('=== 2024 SEASON SUMMARY ===')
print(season_summary[season_summary['year'] == season_summary['year'].max()].sort_values('total_points', ascending=False).to_string(index=False))


Constructor Season Summary: (112, 21)
Columns: ['constructor_name', 'constructor_nationality', 'year', 'total_points', 'avg_finish_position', 'result_volatility', 'finish_rate', 'avg_grid_position', 'avg_position_delta', 'total_races', 'total_laps', 'dnf_count', 'best_finish', 'worst_finish', 'podiums', 'wins', 'championship_position', 'prev_year_points', 'yoy_growth_pct', 'points_per_race', 'momentum_slope']

=== 2024 SEASON SUMMARY ===
constructor_name constructor_nationality  year  total_points  avg_finish_position  result_volatility  finish_rate  avg_grid_position  avg_position_delta  total_races  total_laps  dnf_count  best_finish  worst_finish  podiums  wins  championship_position  prev_year_points  yoy_growth_pct  points_per_race  momentum_slope
         McLaren                 British  2024        609.00                 4.71               3.61         1.00               4.40               -0.31           48        2881          0            1            20       21     6       

## 4. KPI Table 2 — Race-Level Detail
Granular race-by-race data for drill-down in Tableau.


In [4]:
# The cleaned master dataset is already at race-result level
# Add a few more useful columns for Tableau

race_detail = df.copy()

# Add season race number (sequential within year for timeline charts)
race_detail['season_race_num'] = race_detail.groupby('year')['round'].rank(method='dense').astype(int)

# Flag podium finishes
race_detail['is_podium'] = (race_detail['positionOrder'] <= 3).astype(int)
race_detail['is_win'] = (race_detail['positionOrder'] == 1).astype(int)
race_detail['is_points_finish'] = (race_detail['points'] > 0).astype(int)

print(f'Race Detail: {race_detail.shape}')
print(f'New columns added: season_race_num, is_podium, is_win, is_points_finish')


Race Detail: (4626, 32)
New columns added: season_race_num, is_podium, is_win, is_points_finish


## 5. KPI Table 3 — Pit Stop Performance by Constructor & Season


In [5]:
# Merge pit stops with constructor and year info
pit_enriched = pit.merge(
    df[['raceId', 'driverId', 'constructor_name', 'year']].drop_duplicates(),
    on=['raceId', 'driverId'], how='left'
)

# Only normal stops for the summary
normal_pits = pit_enriched[pit_enriched['is_normal_stop'] == 1]

pit_summary = normal_pits.groupby(['constructor_name', 'year']).agg(
    avg_pit_duration=('duration_seconds', 'mean'),
    median_pit_duration=('duration_seconds', 'median'),
    pit_consistency=('duration_seconds', 'std'),
    fastest_stop=('duration_seconds', 'min'),
    slowest_stop=('duration_seconds', 'max'),
    total_stops=('duration_seconds', 'count'),
).reset_index()

# Rank pit stops within each year
pit_summary['pit_rank'] = pit_summary.groupby('year')['avg_pit_duration'].rank().astype(int)

print(f'Pit Stop Summary: {pit_summary.shape}')
print()
print('=== 2024 PIT STOP RANKINGS ===')
latest_year = pit_summary['year'].max()
print(pit_summary[pit_summary['year'] == latest_year].sort_values('avg_pit_duration').to_string(index=False))


Pit Stop Summary: (112, 9)

=== 2024 PIT STOP RANKINGS ===
constructor_name  year  avg_pit_duration  median_pit_duration  pit_consistency  fastest_stop  slowest_stop  total_stops  pit_rank
         McLaren  2024             23.92                22.87             3.57         17.57         36.07           77         1
  Alpine F1 Team  2024             24.22                23.21             3.89         15.79         39.71           71         2
        Mercedes  2024             24.25                22.90             3.85         17.60         37.91           78         3
         Ferrari  2024             24.30                23.08             4.57         17.31         47.19           75         4
        Red Bull  2024             24.36                22.95             4.00         17.22         43.73           80         5
    Aston Martin  2024             24.71                23.27             4.22         17.90         41.67           81         6
      RB F1 Team  2024         

## 6. KPI Table 4 — Sponsor Investment Scorecard
The **ultimate Tableau table**. One row per constructor with a composite "Investment Score" that ranks teams by their attractiveness as a sponsorship target.


In [6]:
# Use the latest 3 seasons for the investment score
recent_years = sorted(season_summary['year'].unique())[-3:]
recent = season_summary[season_summary['year'].isin(recent_years)]

scorecard = recent.groupby('constructor_name').agg(
    avg_points_3yr=('total_points', 'mean'),
    avg_volatility_3yr=('result_volatility', 'mean'),
    avg_finish_rate_3yr=('finish_rate', 'mean'),
    avg_position_delta_3yr=('avg_position_delta', 'mean'),
    latest_momentum=('momentum_slope', 'last'),
    total_podiums_3yr=('podiums', 'sum'),
    total_wins_3yr=('wins', 'sum'),
    latest_championship_pos=('championship_position', 'last'),
).reset_index()

# Normalise each metric to 0-100 scale for the composite score
def normalise(series, invert=False):
    if series.max() == series.min():
        return pd.Series([50] * len(series))
    norm = (series - series.min()) / (series.max() - series.min()) * 100
    return 100 - norm if invert else norm

scorecard['points_score'] = normalise(scorecard['avg_points_3yr'])
scorecard['consistency_score'] = normalise(scorecard['avg_volatility_3yr'], invert=True)  # Lower volatility = better
scorecard['reliability_score'] = normalise(scorecard['avg_finish_rate_3yr'])
scorecard['momentum_score'] = normalise(scorecard['latest_momentum'])
scorecard['race_craft_score'] = normalise(scorecard['avg_position_delta_3yr'])

# Composite Investment Score (weighted)
scorecard['investment_score'] = (
    scorecard['points_score'] * 0.30 +
    scorecard['consistency_score'] * 0.25 +
    scorecard['reliability_score'] * 0.15 +
    scorecard['momentum_score'] * 0.20 +
    scorecard['race_craft_score'] * 0.10
)

scorecard = scorecard.sort_values('investment_score', ascending=False).reset_index(drop=True)
scorecard['investment_rank'] = range(1, len(scorecard) + 1)

# Investment tier
scorecard['tier'] = pd.cut(scorecard['investment_score'], bins=[0, 30, 60, 100],
                           labels=['Bronze — High Risk', 'Silver — Moderate', 'Gold — Premium'])

print('=== SPONSOR INVESTMENT SCORECARD ===')
display_cols = ['investment_rank', 'constructor_name', 'investment_score', 'tier',
                'points_score', 'consistency_score', 'reliability_score', 'momentum_score', 'race_craft_score',
                'latest_championship_pos']
print(scorecard[display_cols].to_string(index=False))


=== SPONSOR INVESTMENT SCORECARD ===
 investment_rank constructor_name  investment_score               tier  points_score  consistency_score  reliability_score  momentum_score  race_craft_score  latest_championship_pos
               1          McLaren             63.02     Gold — Premium         49.68              31.55             100.00          100.00             52.30                        1
               2         Red Bull             55.86  Silver — Moderate        100.00              12.46              84.94            0.00            100.00                        3
               3           Sauber             52.00  Silver — Moderate          0.00             100.00              91.39           28.13             76.64                       10
               4         Mercedes             50.16  Silver — Moderate         63.27              29.93              80.27           19.35             77.87                        4
               5     Aston Martin             41.51  

## 7. Export Tableau-Ready Files
All files are saved to `data/processed/tableau/` for direct import into Tableau Public.


In [7]:
OUT = '../data/processed/tableau/'
os.makedirs(OUT, exist_ok=True)

# Save all 4 KPI tables
season_summary.to_csv(f'{OUT}constructor_season_summary.csv', index=False)
print(f'1. constructor_season_summary.csv  ({season_summary.shape})')

race_detail.to_csv(f'{OUT}race_detail.csv', index=False)
print(f'2. race_detail.csv                 ({race_detail.shape})')

pit_summary.to_csv(f'{OUT}pit_stop_summary.csv', index=False)
print(f'3. pit_stop_summary.csv            ({pit_summary.shape})')

scorecard.to_csv(f'{OUT}investment_scorecard.csv', index=False)
print(f'4. investment_scorecard.csv        ({scorecard.shape})')

print()
print('All Tableau-ready files exported. Ready for dashboard design.')
print()
print('=== FILES SAVED ===')
for f in os.listdir(OUT):
    size = os.path.getsize(os.path.join(OUT, f))
    print(f'  {f:<40} {size/1024:.1f} KB')


1. constructor_season_summary.csv  ((112, 21))
2. race_detail.csv                 ((4626, 32))
3. pit_stop_summary.csv            ((112, 9))
4. investment_scorecard.csv        ((12, 17))

All Tableau-ready files exported. Ready for dashboard design.

=== FILES SAVED ===
  pit_stop_summary.csv                     8.7 KB
  race_detail.csv                          933.7 KB
  investment_scorecard.csv                 2.7 KB
  constructor_season_summary.csv           19.5 KB


## 8. Data Pipeline Complete — Summary

### Files Produced for Tableau:
| File | Rows | Purpose |
|:---|:---|:---|
| `constructor_season_summary.csv` | ~112 | Primary KPI table (1 row per team per season) |
| `race_detail.csv` | ~4,626 | Drill-down (1 row per driver per race) |
| `pit_stop_summary.csv` | ~100 | Pit stop efficiency by team by season |
| `investment_scorecard.csv` | ~10 | Final investment ranking (composite score) |

### Recommended Tableau Dashboards:
1. **Executive Overview** — KPI cards (top team, momentum, volatility) + Points trajectory line chart
2. **Constructor Deep-Dive** — Filter by team to see their full profile (reliability, pit stops, race results)
3. **Investment Scorecard** — Radar chart or bullet chart comparing all teams on 5 investment dimensions

### Next Steps:
- Import these CSVs into Tableau Public
- Build interactive filters (by year, by team, by circuit)
- Publish and share the URL in `tableau/dashboard_links.md`
